# Parse FIX
Resolve one classified message category against the FIX dictionary.

In [ ]:
category = "market"
project_root = "."
source = "logs.messages"
start = None
end = None
fix_dictionary = None
null_values = ["", "null", "<null>", "n/a", "none"]
exclude_msgtypes = ["0", "1"]
protocols = None
fields = None
catalog = {"name": "rekep", "properties": {}}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
merge_by = True
commit_batch_num = 8
commit_row_size = None
limit = None
log_level = "INFO"

In [ ]:
import pyarrow.compute as pc
from pyiceberg.expressions import (
    And,
    GreaterThanOrEqual,
    IsNull,
    LessThan,
    NotIn,
    Or,
)
from rekep.fix.fields import FieldRules
from rekep.fix.registry import FixRegistry
from rekep.fix.rules import MARKET_CATEGORY, MISC_CATEGORY, UNKNOWN_CATEGORY, Rules
from rekep.fix.transcribe import FixCodec
from rekep.iceberg import IcebergCatalog
from rekep.logs import Stage, configure
from rekep.text import FixMsg, Message
from rekep.times import unix_of
from rekep.urls import Url

configure(log_level)
categories = (MARKET_CATEGORY, MISC_CATEGORY, UNKNOWN_CATEGORY)
if category not in categories:
    raise ValueError(f"category must be one of {categories}, got {category!r}")
task_name = f"parse_fix_{category}"
target = f"fix.{category}"
if isinstance(commit_batch_num, bool) or not isinstance(commit_batch_num, int):
    raise TypeError("commit_batch_num must be an integer")
if commit_batch_num <= 0:
    raise ValueError("commit_batch_num must be positive")
if commit_row_size is not None and (
    isinstance(commit_row_size, bool) or not isinstance(commit_row_size, int)
):
    raise TypeError("commit_row_size must be an integer or null")
if commit_row_size is not None and commit_row_size <= 0:
    raise ValueError("commit_row_size must be positive")
if isinstance(exclude_msgtypes, (str, bytes)) or any(
    not isinstance(value, str) for value in exclude_msgtypes
):
    raise TypeError("exclude_msgtypes must be a sequence of strings")
exclude_msgtypes = frozenset(exclude_msgtypes)


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return (
        None
        if not predicates
        else predicates[0]
        if len(predicates) == 1
        else And(*predicates)
    )


protocol_rules = Rules() if protocols is None else Rules.from_dict(protocols)
registry = (
    FixRegistry()
    if fix_dictionary is None
    else FixRegistry(
        cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root),
        announce=print,
    )
)
field_rules = FieldRules() if fields is None else FieldRules.from_dict(fields)
codec = FixCodec(
    rules=protocol_rules,
    registry=registry,
    null_values=frozenset(null_values),
    fields=field_rules,
)
field = FixMsg.into_field()
source_field = Message.into_field()
store = IcebergCatalog.from_dict(catalog)
messages = store.dataset(
    source,
    field=source_field,
    branch=branch,
)
stage = Stage(
    task_name,
    sources={"messages": source},
    targets={category: target},
    window=(unix_of(start), unix_of(end, upper=True)),
)
source_columns = list(
    messages.table_field.names if messages.exists else source_field.names
)
if messages.exists:
    missing = sorted(
        {"msgtype", "entries", "protocol", "eventtype"}
        - set(messages.table_field.names)
    )
    if missing:
        raise ValueError(
            f"{source} is missing {missing}; rebuild it with parse_messages before parsing FIX"
        )

In [ ]:
read = errors = 0
# Every category run owns transaction clock and nesting the Instrument whose
# class maps the canonical
# ticker, so it also says how well it managed: which rung answered for `unix`
# on each row, how many carry an `instrument.symbolticker`, and how many were
# retained with a row-local transcription error. A run
# that hands on a weak base says so here instead of two tables later.
unixsource = {}
tickered = 0


def _measured(batch):
    """One parsed batch, with its clock and ticker coverage counted."""
    global tickered, errors
    for source, count in zip(
        *(
            column.to_pylist()
            for column in pc.value_counts(batch.column("unixsource")).flatten()
        ),
        strict=True,
    ):
        unixsource[source] = unixsource.get(source, 0) + count
    symbolticker = pc.struct_field(batch.column("instrument"), "symbolticker")
    tickered += pc.sum(pc.not_equal(symbolticker, ""), min_count=0).as_py() or 0
    errors += pc.sum(pc.is_valid(batch.column("error")), min_count=0).as_py() or 0
    return batch


# The window is read off the *stored* recording clock, because that is what
# the message stage partitioned on. `unix` moves when a transaction time
# resolves, so filtering on it here would drop rows the interval owns.
lower, upper = unix_of(start), unix_of(end, upper=True)
selection = _window(lower, upper)
if exclude_msgtypes:
    # Null discriminators still need best-effort transcription. Only named
    # session liveness traffic is left in logs.messages by this stage.
    application_messages = Or(IsNull("msgtype"), NotIn("msgtype", exclude_msgtypes))
    selection = (
        application_messages
        if selection is None
        else And(selection, application_messages)
    )
# Each parallel run pushes its complete category predicate into Iceberg. A
# row outside it is never transcribed, enriched, buffered or written here.
category_events = protocol_rules.into_iceberg_category_filter(
    category, registry.versions
)
selection = category_events if selection is None else And(selection, category_events)


output = store.dataset(
    target,
    field=field,
    table_properties=dict(table_properties),
    branch=branch,
    commit_batch_num=commit_batch_num,
    commit_row_size=commit_row_size,
)


def _batches():
    global read
    for staged in messages.read_arrow_reader(
        columns=source_columns, row_filter=selection
    ):
        if limit is not None and read + staged.num_rows > limit:
            staged = staged.slice(0, max(0, limit - read))
        if not staged.num_rows:
            continue
        read += staged.num_rows
        batch = _measured(FixMsg.from_message_batch(staged, codec))
        yield batch
        if limit is not None and read >= limit:
            break


written = output.append_arrow_reader(
    _batches(),
    field,
    merge_by=merge_by,
    commit_row_size=commit_row_size,
    commit_batch_num=commit_batch_num,
)
skipped = read - written

In [ ]:
stage.says("selected %d %s rows", read, category)
# The base the next two stages key on, said where it was built rather than
# discovered two tables later.
stage.says(
    "resolved unix from %s; %d of %d rows carry a symbolticker",
    ", ".join(f"{rung} {count}" for rung, count in sorted(unixsource.items()))
    or "nothing",
    tickered,
    read,
)
stage.says("retained %d rows with FIX transcription errors", errors)
result = stage.finished(
    read=read,
    written=written,
    skipped=skipped,
    category=category,
    unixsource=unixsource,
    tickered=tickered,
    errors=errors,
)
result